# 🧠 CerebroSpheroid V5: AIKosh NVIDIA A100 (5GB MIG) Engine
### 105.9M V5 Small Architecture • 100% Loss-Bearing Contiguous Pretraining • 5GB VRAM Tuned

- **Pure JAX on NVIDIA A100 (1g.5gb MIG)**: Native Ampere `bfloat16` Tensor Core acceleration with `XLA_PYTHON_CLIENT_PREALLOCATE=false` to stay safely within 5 GB VRAM.
- **5GB-Safe Auto-Tuning**: Caps micro-batch (`max_batch_size=4`) with activation rematerialization (`remat="true"`) while preserving `effective_batch_size=64` via gradient accumulation.
- **4-Phase 41.4B Pretraining Curriculum**: `15.7B` FineWeb/Code → `15.7B` FineWeb-Edu/Code/Math → `5.0B` FineWeb-Edu/Code/Math → `5.0B` FineWeb-Edu/Code/Math/Cosmopedia.
- **Remote Cloud Sync**: Automatically fetches `HF_TOKEN` from `bioflo.in`, resumes `v5_small_latest.pt`, and streams checkpoints + loss logs back to `bioflo.in`.

## ⚡ Step 1: AIKosh A100 5GB Environment Setup & Vault Auth
Run with **Shift + Enter**. Verifies the A100 CUDA GPU, configures 5GB-safe XLA memory growth, pulls `origin/main`, and authenticates Hugging Face + `bioflo.in`.

In [ ]:
# ==============================================================================
# 🚀 ONE-CLICK SETUP: CEREBROSPHEROID V5 (AIKOSH A100 5GB MIG)
# ==============================================================================
import os, sys, subprocess, importlib

# Must be set BEFORE importing jax so XLA does not preallocate 75%+ of the 5GB MIG slice
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["TF_FORCE_GPU_ALLOW_GROWTH"] = "true"
os.environ.pop("PJRT_DEVICE", None)

# 1. Verify NVIDIA GPU & install dependencies
subprocess.run(["nvidia-smi"], check=False)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "transformers", "datasets", "sentencepiece", "optax", "torch", "pyarrow", "huggingface_hub", "certifi"],
    check=True,
)

try:
    import jax
    if jax.default_backend().lower() != "gpu":
        raise RuntimeError(f"Expected GPU backend, got {jax.default_backend()}")
except Exception as _jax_err:
    print(f"⚠️ JAX CUDA backend not active ({_jax_err}); installing jax[cuda12]...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "jax[cuda12]"], check=True)
    import jax

print("JAX backend:", jax.default_backend().upper())
print("GPU devices:", jax.devices())
if jax.default_backend().lower() != "gpu":
    raise RuntimeError("This notebook requires an NVIDIA GPU runtime (AIKosh A100 5GB MIG).")

# 2. Synchronize Repository to latest origin/main
BASE_DIR = os.path.expanduser("~")
REPO_DIR = os.path.join(BASE_DIR, "cerebro_v5")
if not os.path.exists(REPO_DIR):
    print("📥 Cloning cerebro_spheroid_v5 repository...")
    subprocess.run(["git", "clone", "https://github.com/AjitKumar-git/cerebro_spheroid_v5.git", REPO_DIR], check=True)
else:
    print("🔄 Pulling latest updates from origin/main...")
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--all"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "origin/main"], check=True)

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

importlib.invalidate_caches()
for mod in list(sys.modules.keys()):
    if mod.startswith(("src", "training", "autotune", "colab")):
        del sys.modules[mod]

# 3. Checkpoint Storage Setup
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints", "v5_small_scratch")
os.makedirs(CKPT_DIR, exist_ok=True)
print(f"📁 Checkpoint directory: {CKPT_DIR}")

# 4. Dynamic Token & Credential Resolution from bioflo.in Vault
from src.cloud_sync import get_sync_token
from src.hf_utils import get_hf_token

sync_token = get_sync_token()
os.environ["BIOFLO_SYNC_TOKEN"] = sync_token
os.environ["CEREBRO_SYNC_TOKEN"] = sync_token

hf_tok = get_hf_token()
if hf_tok:
    os.environ["HF_TOKEN"] = hf_tok
    os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_tok
    try:
        from huggingface_hub import login
        login(token=hf_tok, add_to_git_credential=False)
        print("🔑 Hugging Face Hub successfully authenticated dynamically from website vault.")
    except Exception as e_login:
        print(f"ℹ️ Hugging Face login notice: {e_login}")
else:
    print("ℹ️ Hugging Face token not found in website vault.")


## Step 2: Configure & Launch 5GB-Tuned Pretraining (`v5_small`)
- `max_batch_size=4` + `remat="true"`: keeps peak activation + optimizer VRAM inside the **5 GB A100 MIG** envelope.
- `effective_batch_size=64`: maintains exact gradient-accumulation parity with the TPU run.
- `TARGET_TOKENS = 41_400_000_000`: 4-phase pretraining (`15.7B` + `15.7B` + `5.0B` + `5.0B`), resuming `latest.pt` from `bioflo.in`.

In [ ]:
# ==============================================================================
# ⚡ Step 2: Launch CerebroSpheroid V5 Small on AIKosh A100 5GB
# ==============================================================================
import os, sys, importlib

BASE_DIR = os.path.expanduser("~")
REPO_DIR = os.path.join(BASE_DIR, "cerebro_v5")
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints", "v5_small_scratch")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
os.chdir(REPO_DIR)

importlib.invalidate_caches()
for mod in list(sys.modules.keys()):
    if mod.startswith(("src", "training", "autotune", "colab")):
        del sys.modules[mod]

from src.cloud_sync import get_sync_token
from training.train_v5_tpu_jax import run_training

# ── Primary Configuration (A100 5GB MIG) ──────────────────────────────────────
MODEL_PRESET       = 'v5_small'               # 105.9M Parameters (16 Shells, 40 Shortcut Tracts)
FROM_SCRATCH       = False                    # Resume latest checkpoint from bioflo.in / local disk
AUTO_TUNE          = True                     # Auto-tune tests micro-batches [1, 2, 4] safely under 5GB VRAM
FORCE_TUNE         = False
MAX_MICRO_BATCH    = 4                        # Hard ceiling for 5GB A100 MIG (increase only on larger GPUs)
EFFECTIVE_BATCH    = 64                       # Global effective batch size via gradient accumulation
TARGET_TOKENS      = 41_400_000_000          # 41.4B Pretrain Budget (15.7B P0 + 15.7B P1 + 5B P2 + 5B P3)
DATASET_SOURCE     = 'pretrain'               # 4-phase pretraining curriculum (FineWeb -> FineWeb-Edu P1+)
THINKING_CYCLES    = 4                        # Subconscious deliberation cycles per token
STAGE_STEPS        = 100_000                  # Context steps: 512 -> 1024 -> 2048 -> 4096
MIN_WINDOW         = 512                      # Minimum initial context window

run_training(
    preset=MODEL_PRESET,
    from_scratch=FROM_SCRATCH,
    auto_tune=AUTO_TUNE,
    force_tune=FORCE_TUNE,
    max_batch_size=MAX_MICRO_BATCH,
    remat="true",                             # Activation checkpointing required for 5GB VRAM
    dtype="auto",                             # Auto-selects native A100 bfloat16 Tensor Cores
    effective_batch_size=EFFECTIVE_BATCH,
    target_tokens=TARGET_TOKENS,
    dataset_source=DATASET_SOURCE,
    elastic_window=False,
    thinking_cycles=THINKING_CYCLES,
    min_context_window=MIN_WINDOW,
    context_stage_steps=STAGE_STEPS,
    total_window=4096,
    sensory_position_steps=4096,
    max_phase_steps=4096,
    checkpoint_dir=CKPT_DIR,
    cloud_sync=True,
    cloud_token=get_sync_token(),
    cloud_weights_only=True,
    save_optimizer_state=True,
    scratch_dir="/tmp/cerebro_v5_ckpts",
    resume_target="latest",
    scmer_lambda=0.05,
    scmer_margin=0.12,
    scmer_subsample=512,
    scmer_pair_samples=8192,
    scmer_interval=8,
    selective_loss=True,
    selective_focal_gamma=0.5,
    selective_noise_cap=8.5,
    z_loss_weight=1e-4,
)


## Step 3: Monitor Checkpoints & Cloud Synchronization
Inspect local checkpoints and live sync status on `bioflo.in`.

In [ ]:
# ==============================================================================
# 📊 Step 3: Checkpoint Inspector & Cloud Hub Dashboard
# ==============================================================================
import os, json
from src.cloud_sync import check_remote_manifest, get_sync_token

CKPT_DIR = os.path.expanduser("~/checkpoints/v5_small_scratch")
print(f"📁 Local Saved Checkpoints ({CKPT_DIR}):")
if os.path.exists(CKPT_DIR):
    for fn in sorted(os.listdir(CKPT_DIR)):
        fp = os.path.join(CKPT_DIR, fn)
        if os.path.isfile(fp):
            print(f"  {fn}: {os.path.getsize(fp)/1e6:.2f} MB")

print("\n🌐 Remote bioflo.in Cloud Hub Status (v5_small):")
try:
    manifest = check_remote_manifest(model="v5_small", token=get_sync_token())
    print(json.dumps(manifest, indent=2))
except Exception as e:
    print(f"ℹ️ Cloud sync status notice: {e}")

print("\n📈 Live Web Dashboard: https://bioflo.in/cerebro")


## Step 4: Interactive Live Inference & Prompt Testing
Generate completions from your trained `v5_small` checkpoint (locally cached or auto-pulled from `bioflo.in`).

In [ ]:
# ==============================================================================
# 💬 Step 4: Interactive Live Inference & Prompt Testing
# ==============================================================================
import os, jax
import jax.numpy as jnp
import numpy as np
from src.jax.v5_model import ModelConfigV5, PRESETS_V5, training_rollout
from src.jax.checkpoint import load_torch_pt_to_jax, discover_latest_checkpoint
from src.cloud_sync import pull_checkpoint
from src.corpus import build_tokenizer, format_stimulus_prompt, clean_motor_response, encode_no_special

CKPT_DIR = os.path.expanduser("~/checkpoints/v5_small_scratch")
CKPT_PATH = os.path.join(CKPT_DIR, "v5_small_best.pt")
if not os.path.exists(CKPT_PATH):
    CKPT_PATH = os.path.join(CKPT_DIR, "v5_small_latest.pt")
if not os.path.exists(CKPT_PATH):
    disc_path, _, _ = discover_latest_checkpoint(CKPT_DIR, preset="v5_small")
    if disc_path and os.path.exists(disc_path):
        CKPT_PATH = disc_path
    else:
        print("🌐 Checkpoint not found locally. Pulling latest.pt from bioflo.in...")
        pull_checkpoint(CKPT_PATH, target_type="latest", model="v5_small")

print(f"Loading checkpoint: {CKPT_PATH}...")
tokenizer = build_tokenizer("llama")

cfg_dict = PRESETS_V5["v5_small"].copy()
cfg_dict["thinking_cycles"] = 4
cfg_dict["use_adaptive_thinking"] = False
cfg_dict["sensory_position_steps"] = 4096
cfg_dict["max_phase_steps"] = 4096
cfg_dict["max_seq_len"] = 4096
config = ModelConfigV5(**cfg_dict)

params, buffers, meta = load_torch_pt_to_jax(CKPT_PATH, config, target_dtype=jnp.bfloat16)
print(f"✓ Loaded model from Step {meta.get('step', 0):,} (Loss: {meta.get('loss', 0.0):.4f})")

def generate(prompt_text: str, max_tokens: int = 128) -> str:
    stimulus = format_stimulus_prompt(prompt_text)
    tokens = encode_no_special(tokenizer, stimulus)
    prompt_arr = jnp.array([tokens], dtype=jnp.int32)

    out_logits = training_rollout(
        params=params,
        prompts=prompt_arr,
        target_len=max_tokens,
        buffers=buffers,
        config=config,
        targets=None,
        feedback_tokens=None,
        pure_teacher_forcing=False,
        deliberation_mode="teacher_forcing",
        repetition_penalty=1.15,
    )
    gen_token_ids = np.array(jnp.argmax(out_logits, axis=-1)[0])
    raw = tokenizer.decode(gen_token_ids, skip_special_tokens=False)
    return clean_motor_response(raw).strip()

test_prompt = "What is the capital of France?"
print(f"\nPrompt:   {test_prompt}")
print(f"Response: {generate(test_prompt)}")
